<a href="https://colab.research.google.com/github/marcusvbrangel/production-surveillance/blob/main/production_surveillance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production Surveillance

### Dicionário de Dados — Volve Field Daily Data

| Coluna | Descrição | Tipo de Dados | Unidade de Medida | Equipamento / Origem da Medição | Local da Medição |
|---|---|---|---|---|---|
| `DATEPRD` | Data da produção/operação diária | datetime | data | Sistema operacional / historian | Centro de supervisão / banco operacional |
| `NPD_WELL_BORE_CODE` | Código oficial do poço | inteiro | N/A | Cadastro corporativo | Sistema corporativo |
| `NPD_FIELD_CODE` | Código oficial do campo | inteiro | N/A | Cadastro corporativo | Sistema corporativo |
| `NPD_FIELD_NAME` | Nome do campo petrolífero | string | N/A | Cadastro corporativo | Sistema corporativo |
| `NPD_FACILITY_CODE` | Código da instalação offshore | inteiro | N/A | Cadastro corporativo | Sistema corporativo |
| `NPD_FACILITY_NAME` | Nome da instalação/FPSO/plataforma | string | N/A | Cadastro corporativo | Sistema corporativo |
| `WELL_BORE_CODE` | Código interno do poço | inteiro | N/A | Cadastro corporativo | Sistema corporativo |
| `WELL_BORE_NAME` | Nome do poço | string | N/A | Cadastro corporativo | Identificação operacional do poço |
| `WELL_FIELD_CODE` | Código do campo associado ao poço | inteiro | N/A | Cadastro corporativo | Sistema corporativo |
| `WELL_FIELD_NAME` | Nome do campo associado ao poço | string | N/A | Cadastro corporativo | Sistema corporativo |
| `WELL_TYPE` | Tipo do poço (produtor/injetor) | string | N/A | Engenharia de produção | Configuração operacional do poço |
| `FLOW_KIND` | Tipo de fluxo/operação | string | N/A | Sistema operacional | Linha operacional do poço |
| `ON_STREAM_HRS` | Quantidade de horas produzindo no dia | float | horas | Sistema supervisório / produção | Status operacional do poço |
| `AVG_DOWNHOLE_PRESSURE` | Pressão média no fundo do poço | float | bar(a) | Gauge de fundo / sensor downhole | Fundo do poço / próximo da zona produtora |
| `AVG_DOWNHOLE_TEMPERATURE` | Temperatura média no fundo do poço | float | °C | Sensor downhole | Fundo do poço / tubing inferior |
| `AVG_DP_TUBING` | Delta de pressão médio no tubing | float | bar | Sensores de pressão tubing | Interior do tubing de produção |
| `AVG_ANNULUS_PRESS` | Pressão média do anular | float | bar | Sensor anular | Espaço anular entre casing e tubing |
| `AVG_CHOKE_SIZE_P` | Abertura média do choke | float | % | Atuador/sensor do choke | Choke na árvore de natal / superfície |
| `AVG_WHP_P` | Pressão média na cabeça do poço | float | bar | Sensor wellhead | Cabeça do poço / árvore de natal |
| `AVG_WHT_P` | Temperatura média na cabeça do poço | float | °C | Sensor wellhead | Cabeça do poço / árvore de natal |
| `DP_CHOKE_SIZE` | Delta de pressão associado ao choke | float | bar | Sensores upstream/downstream choke | Antes e depois do choke |
| `BORE_OIL_VOL` | Volume diário de óleo produzido | float | Sm3/d | Medidor multifásico / teste produção | Linha de produção do poço |
| `BORE_GAS_VOL` | Volume diário de gás produzido | float | Sm3/d | Medidor de gás | Linha de gás / separador |
| `BORE_WAT_VOL` | Volume diário de água produzida | float | Sm3/d | Medidor multifásico | Linha de produção / separador |
| `BORE_WI_VOL` | Volume diário de água injetada | float | Sm3/d | Medidor de injeção | Linha de injeção de água |

---
# FASE 01 — LEITURA E QA/QC

Aprender:

- parse temporal
- índices temporais
- ordenação temporal
- missing data
- consistência temporal
---

In [293]:
# importacao das bibliotecas
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [294]:
# caminho do arquivo
base_dir = Path.cwd()
if base_dir.name != "10-exercicio-production-surveillance":
    base_dir = base_dir / "notebooks" / "10-exercicio-production-surveillance"
arquivo = base_dir / "volve-field-daily-data.xlsx"

In [295]:
# leitura do arquivo excel
df_raw = pd.read_excel(arquivo)

In [296]:
# visualizacao inicial
df_raw.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [297]:
# dimensao do dataset
df_raw.shape

(15634, 24)

In [298]:
# informacoes gerais: nome, nulos e tipos
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 15634 entries, 0 to 15633
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   DATEPRD                   15634 non-null  datetime64[us]
 1   WELL_BORE_CODE            15634 non-null  str           
 2   NPD_WELL_BORE_CODE        15634 non-null  int64         
 3   NPD_WELL_BORE_NAME        15634 non-null  str           
 4   NPD_FIELD_CODE            15634 non-null  int64         
 5   NPD_FIELD_NAME            15634 non-null  str           
 6   NPD_FACILITY_CODE         15634 non-null  int64         
 7   NPD_FACILITY_NAME         15634 non-null  str           
 8   ON_STREAM_HRS             15349 non-null  float64       
 9   AVG_DOWNHOLE_PRESSURE     8980 non-null   float64       
 10  AVG_DOWNHOLE_TEMPERATURE  8980 non-null   float64       
 11  AVG_DP_TUBING             8980 non-null   float64       
 12  AVG_ANNULUS_PRESS         789

In [299]:
# ============================================================
# 1. Parse temporal
# ============================================================

# converter a coluna de data para o tipo datetime
df_raw["DATEPRD"] = pd.to_datetime(df_raw["DATEPRD"], errors="coerce")

In [300]:
# verificacao do intervalo temporal geral
print("Data inicial:", df_raw["DATEPRD"].min())
print("Data final:", df_raw["DATEPRD"].max())

Data inicial: 2007-09-01 00:00:00
Data final: 2016-12-01 00:00:00


In [301]:
# verificacao se houve erro na conversao de data
datas_invalidas = df_raw["DATEPRD"].isna().sum()

print("Quantidade de datas invalidas:", datas_invalidas)

Quantidade de datas invalidas: 0


In [302]:
# ============================================================
# 2. Ordenação temporal
# ============================================================

# ordenar por poco e data
df_raw = df_raw.sort_values(
    by=["NPD_WELL_BORE_NAME", "DATEPRD"]
).reset_index(drop=True)

df_raw.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,WI
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,%,0.00000,0.00000,0.00000,0.0,0.0,0.0,NaN,production,OP
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,%,33.09788,10.47992,33.07195,0.0,0.0,0.0,NaN,production,OP


In [303]:
# verificacao dos pocos existente
df_raw["NPD_WELL_BORE_NAME"].unique()

<ArrowStringArray>
[ '15/9-F-1 C',   '15/9-F-11',   '15/9-F-12',   '15/9-F-14', '15/9-F-15 D',
    '15/9-F-4',    '15/9-F-5']
Length: 7, dtype: str

In [304]:
# quantidade de registros por poco
df_raw["NPD_WELL_BORE_NAME"].value_counts()

NPD_WELL_BORE_NAME
15/9-F-4       3327
15/9-F-5       3306
15/9-F-12      3056
15/9-F-14      3056
15/9-F-11      1165
15/9-F-15 D     978
15/9-F-1 C      746
Name: count, dtype: int64

In [305]:
# intervalo temporal por poco
intervalo_por_poco = (
    df_raw.groupby("NPD_WELL_BORE_NAME")
        .agg(
            data_inicial=("DATEPRD", "min"),
            data_final=("DATEPRD", "max"),
            qtd_registros=("DATEPRD", "count")
        )
        .sort_values("data_inicial")
)

intervalo_por_poco

,data_inicial,data_final,qtd_registros
NPD_WELL_BORE_NAME,,,
15/9-F-4,2007-09-01,2016-12-01,3327
15/9-F-5,2007-09-01,2016-09-18,3306
15/9-F-12,2008-02-12,2016-09-17,3056
15/9-F-14,2008-02-12,2016-09-17,3056
15/9-F-11,2013-07-08,2016-09-17,1165
15/9-F-15 D,2014-01-12,2016-09-17,978
15/9-F-1 C,2014-04-07,2016-04-21,746


In [306]:
# ============================================================
# 3. Missing data
# ============================================================

missing = (
    df_raw.isna()
    .sum()
    .to_frame("qtd_missing")
)

In [307]:
missing["percent_missing"] = (
    missing["qtd_missing"] / len(df_raw) * 100
).round(2)

missing.sort_values("percent_missing", ascending=False)

,qtd_missing,percent_missing
BORE_WI_VOL,9928,63.50
AVG_ANNULUS_PRESS,7744,49.53
AVG_CHOKE_SIZE_P,6715,42.95
AVG_DOWNHOLE_PRESSURE,6654,42.56
AVG_DOWNHOLE_TEMPERATURE,6654,42.56
AVG_DP_TUBING,6654,42.56
AVG_WHT_P,6488,41.50
AVG_WHP_P,6479,41.44
BORE_WAT_VOL,6473,41.40
AVG_CHOKE_UOM,6473,41.40


In [308]:
# missing por poco nas principais variaveis de producao
colunas_principais = [
    "ON_STREAM_HRS",
    "AVG_DOWNHOLE_PRESSURE",
    "AVG_DOWNHOLE_TEMPERATURE",
    "AVG_CHOKE_SIZE_P",
    "AVG_WHP_P",
    "AVG_WHT_P",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
]

In [309]:
missing_por_poco = (
    df_raw.groupby("NPD_WELL_BORE_NAME")[colunas_principais]
        .apply(lambda x: x.isna().mean() * 100)
        .round(2)
)

missing_por_poco

,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,AVG_CHOKE_SIZE_P,AVG_WHP_P,AVG_WHT_P,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
NPD_WELL_BORE_NAME,,,,,,,,,
15/9-F-1 C,0.00,0.40,0.40,0.00,0.00,0.00,0.00,0.00,0.00
15/9-F-11,0.00,0.52,0.52,0.17,0.52,0.52,0.00,0.00,0.00
15/9-F-12,0.00,0.20,0.20,1.44,0.00,0.00,0.00,0.00,0.00
15/9-F-14,0.00,0.20,0.20,6.41,0.00,0.00,0.00,0.00,0.00
15/9-F-15 D,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
15/9-F-4,4.57,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
15/9-F-5,4.02,100.00,100.00,95.16,95.16,95.43,95.16,95.16,95.16


In [310]:
# ============================================================
# 4. Consistência temporal
# ============================================================

# verificar datas duplicadas por poco
duplicadas = (
    df_raw.duplicated(subset=["NPD_WELL_BORE_NAME", "DATEPRD"])
        .sum()
)

print("Quantidade de registros duplicados por poco/data:", duplicadas)

Quantidade de registros duplicados por poco/data: 0


In [311]:
# mostrar duplicadas, se existirem
df_raw[df_raw.duplicated(subset=["NPD_WELL_BORE_NAME", "DATEPRD"], keep=False)]
#

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,AVG_CHOKE_UOM,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE


In [312]:
# verificar gaps temporais por poco
# como o dataset e diario, esperamos diferenca de 1 dia entre os registros consecutivos
df_raw["diff_dias"] = (
    df_raw.groupby("NPD_WELL_BORE_NAME")["DATEPRD"]
        .diff()
        .dt.days
)

gaps = df_raw[df_raw["diff_dias"] > 1][
        ["NPD_WELL_BORE_NAME", "DATEPRD", "diff_dias"]
    ]

gaps.head(20)

,NPD_WELL_BORE_NAME,DATEPRD,diff_dias
819,15/9-F-11,2013-09-20,2.0
951,15/9-F-11,2014-01-31,2.0
958,15/9-F-11,2014-02-08,2.0
1987,15/9-F-12,2008-04-29,2.0
2087,15/9-F-12,2008-08-08,2.0
2357,15/9-F-12,2009-05-06,2.0
2446,15/9-F-12,2009-08-04,2.0
2474,15/9-F-12,2009-09-02,2.0
2653,15/9-F-12,2010-03-02,3.0
2658,15/9-F-12,2010-03-08,2.0


In [313]:
# resumo de gaps por poco
resumo_gaps = (
    gaps.groupby("NPD_WELL_BORE_NAME")
        .agg(
            qtd_gaps=("diff_dias", "count"),
            maior_gap_dias=("diff_dias", "max"),
            media_gap_dias=("diff_dias", "mean")
        )
        .round(2)
)

resumo_gaps

,qtd_gaps,maior_gap_dias,media_gap_dias
NPD_WELL_BORE_NAME,,,
15/9-F-11,3,2.0,2.00
15/9-F-12,46,12.0,2.85
15/9-F-14,46,12.0,2.85
15/9-F-15 D,2,2.0,2.00
15/9-F-4,2,30.0,27.50


In [314]:
# ============================================================
# 5. Consistência básica dos valores
# ============================================================

# estatistica descritiva das variaveis numericas
df_raw.describe().T

,count,mean,min,25%,50%,75%,max,std
DATEPRD,15634,2012-11-07 17:39:58.004349,2007-09-01 00:00:00,2010-07-30 00:00:00,2013-05-08 00:00:00,2015-02-19 00:00:00,2016-12-01 00:00:00,NaN
NPD_WELL_BORE_CODE,15634.0,5908.581745,5351.0,5599.0,5693.0,5769.0,7405.0,649.231622
NPD_FIELD_CODE,15634.0,3420717.0,3420717.0,3420717.0,3420717.0,3420717.0,3420717.0,0.0
NPD_FACILITY_CODE,15634.0,369304.0,369304.0,369304.0,369304.0,369304.0,369304.0,0.0
ON_STREAM_HRS,15349.0,19.994093,0.0,24.0,24.0,24.0,25.0,8.369978
AVG_DOWNHOLE_PRESSURE,8980.0,181.803869,0.0,0.0,232.896939,255.401455,397.58855,109.712363
AVG_DOWNHOLE_TEMPERATURE,8980.0,77.162969,0.0,0.0,103.186689,106.276591,108.502178,45.657948
AVG_DP_TUBING,8980.0,154.028787,0.0,83.665361,175.588861,204.319964,345.90677,76.752373
AVG_ANNULUS_PRESS,7890.0,14.8561,0.0,10.841437,16.308598,21.306125,30.019828,8.406822
AVG_CHOKE_SIZE_P,8919.0,55.168533,0.0,18.952989,52.096877,99.924288,100.0,36.692924


In [315]:
# verificar valores negativo em variaveis que, em tese, nao deveriam ser negativas
coluna_nao_negativas = [
    "ON_STREAM_HRS",
    "AVG_CHOKE_SIZE_P",
    "BORE_OIL_VOL",
    "BORE_GAS_VOL",
    "BORE_WAT_VOL",
    "BORE_WI_VOL",
]

for col in coluna_nao_negativas:
    qtd_negativos = (df_raw[col] < 0).sum()
    print(f"{col}: {qtd_negativos} valores negativos")

ON_STREAM_HRS: 0 valores negativos
AVG_CHOKE_SIZE_P: 0 valores negativos
BORE_OIL_VOL: 0 valores negativos
BORE_GAS_VOL: 0 valores negativos
BORE_WAT_VOL: 4 valores negativos
BORE_WI_VOL: 0 valores negativos


In [316]:
# verificar producao com poco teoricamente offline
mask = ((df_raw["ON_STREAM_HRS"] == 0) &
       (
           (df_raw["BORE_OIL_VOL"] > 0) |
           (df_raw["BORE_GAS_VOL"] > 0) |
           (df_raw["BORE_WAT_VOL"] > 0)
       )
)

offline_com_producao = df_raw[mask]

offline_com_producao[
    ["DATEPRD", "NPD_WELL_BORE_NAME", "ON_STREAM_HRS", "BORE_OIL_VOL", "BORE_GAS_VOL", "BORE_WAT_VOL"]
].head(20)

,DATEPRD,NPD_WELL_BORE_NAME,ON_STREAM_HRS,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL
1301,2015-01-17,15/9-F-11,0.0,1026.57,150773.50,461.02
3283,2011-12-26,15/9-F-12,0.0,0.00,7.09,9.71
6339,2011-12-26,15/9-F-14,0.0,0.00,56.38,45.12


In [317]:
# verificar linhas com producao de oleo positiva, mas com horas online zeradas ou ausentes
mask = (
    (df_raw["BORE_OIL_VOL"] > 0) &
    (
        (df_raw["ON_STREAM_HRS"] <= 0) |
        (df_raw["ON_STREAM_HRS"].isna())
    )
)

problema_horas = df_raw[mask]

problema_horas[
    ["DATEPRD", "NPD_WELL_BORE_NAME", "ON_STREAM_HRS", "BORE_OIL_VOL"]
].head(20)

,DATEPRD,NPD_WELL_BORE_NAME,ON_STREAM_HRS,BORE_OIL_VOL
1301,2015-01-17,15/9-F-11,0.0,1026.57


---
# FASE 03 — FEATURE ENGINEERING TEMPORAL
---

In [318]:
df_well = df_raw.copy()

In [319]:
df_well.shape

(15634, 25)

In [320]:
df_well.set_index("DATEPRD", inplace=True)

In [321]:
df_well.sample(10)

,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,...,AVG_WHP_P,AVG_WHT_P,DP_CHOKE_SIZE,BORE_OIL_VOL,BORE_GAS_VOL,BORE_WAT_VOL,BORE_WI_VOL,FLOW_KIND,WELL_TYPE,diff_dias
DATEPRD,,,,,,,,,,,,,,,,,,,,,
2011-05-27,NO 15/9-F-4 AH,5693,15/9-F-4,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,6009.187415,injection,WI,1.0
2013-11-03,NO 15/9-F-11 H,7078,15/9-F-11,3420717,VOLVE,369304,MÆRSK INSPIRER,17.33333,256.930342,105.881719,...,81.737068,55.919774,53.533226,773.31,118945.98,45.93,NaN,production,OP,1.0
2013-12-29,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,5911.186745,injection,WI,1.0
2009-10-05,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,23.16000,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,7440.000000,injection,WI,1.0
2015-01-08,NO 15/9-F-15 D,7289,15/9-F-15 D,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,214.062555,106.947410,...,48.556286,32.097432,19.708515,225.72,32736.70,6.99,NaN,production,OP,1.0
2008-08-30,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,3206.858773,injection,WI,1.0
2015-06-25,NO 15/9-F-14 H,5351,15/9-F-14,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,267.513339,99.552869,...,30.779034,88.990955,2.157189,199.69,29406.13,3512.97,NaN,production,OP,1.0
2016-01-07,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,NaN,NaN,...,NaN,NaN,0.000000,NaN,NaN,NaN,6886.851013,injection,WI,1.0
2013-09-26,NO 15/9-F-14 H,5351,15/9-F-14,3420717,VOLVE,369304,MÆRSK INSPIRER,24.00000,247.787211,100.483648,...,32.463411,88.934755,2.950250,661.93,98839.06,3691.26,NaN,production,OP,1.0


In [322]:
# --------------------------------------
# 01 - lag feature
# --------------------------------------
# memoria temporal
# o modelo aprende comportamento pasado
# --------------------------------------

lag_features = [1, 3, 7, 14, 30]

for lag in lag_features:

    df_well[f"oil_lag_{lag}"] = (
        df_well["BORE_OIL_VOL"]
        .shift(lag)
    )

    df_well[f"gas_lag_{lag}"] = (
        df_well["BORE_GAS_VOL"]
        .shift(lag)
    )

    df_well[f"water_lag_{lag}"] = (
        df_well["BORE_WAT_VOL"]
        .shift(lag)
    )

In [323]:
# --------------------------------------
# 02 - rolling mean
# --------------------------------------
# tendencia operacional
# remove ruido
# --------------------------------------

rolling_windows = [3, 7, 14, 30]

for window in rolling_windows:

    df_well[f"oil_roll_mean_{window}"] = (
        df_well["BORE_OIL_VOL"]
        .rolling(window=window)
        .mean()
    )

    df_well[f"gas_roll_mean_{window}"] = (
        df_well["BORE_GAS_VOL"]
        .rolling(window=window)
        .mean()
    )

    df_well[f"water_roll_mean_{window}"] = (
        df_well["BORE_WAT_VOL"]
        .rolling(window=window)
        .mean()
    )

In [324]:
# --------------------------------------
# 03 - rolling std
# --------------------------------------
# estabilidade operacional
# volatilidade
# --------------------------------------

std_windows = [7, 14, 30]

for window in std_windows:

    df_well[f"oil_roll_std_{window}"] = (
        df_well["BORE_OIL_VOL"]
        .rolling(window=window)
        .std()
    )

    df_well[f"gas_roll_std_{window}"] = (
        df_well["BORE_GAS_VOL"]
        .rolling(window=window)
        .std()
    )

    df_well[f"water_roll_std_{window}"] = (
        df_well["BORE_WAT_VOL"]
        .rolling(window=window)
        .std()
    )

In [325]:
# --------------------------------------
# 04 - deltas
# --------------------------------------
# mudanca absoluta
# velocidade operacional
# --------------------------------------

delta_periods = [1, 3, 7]

for period in delta_periods:

    df_well[f"oil_delta_{period}d"] = (
        df_well["BORE_OIL_VOL"] - df_well["BORE_OIL_VOL"].shift(period)
    )

    df_well[f"gas_delta_{period}d"] = (
        df_well["BORE_GAS_VOL"] - df_well["BORE_GAS_VOL"].shift(period)
    )

    df_well[f"water_delta_{period}d"] = (
        df_well["BORE_WAT_VOL"] - df_well["BORE_WAT_VOL"].shift(period)
    )


In [326]:
# --------------------------------------
# 05 - diferencas percentuais
# --------------------------------------
# degradacao relativa
# --------------------------------------

pct_periods = [1, 7, 14]

for period in pct_periods:

    df_well[f"oil_pct_change_{period}d"] = (
        df_well["BORE_OIL_VOL"]
        .pct_change(periods=period)
    )

    df_well[f"gas_pct_change_{period}d"] = (
        df_well["BORE_GAS_VOL"]
        .pct_change(periods=period)
    )

    df_well[f"water_pct_change_{period}d"] = (
        df_well["BORE_WAT_VOL"]
        .pct_change(periods=period)
    )

In [327]:
# --------------------------------------
# 06 - ewma
# --------------------------------------
# exponencial weighted moving average
# maior peso para dados recentes
# --------------------------------------

ewma_spans = [3, 7, 14, 30]

for span in ewma_spans:

    df_well[f"oil_ewma_{span}"] = (
        df_well["BORE_OIL_VOL"]
        .ewm(span=span)
        .mean()
    )

    df_well[f"gas_ewma_{span}"] = (
        df_well["BORE_GAS_VOL"]
        .ewm(span=span)
        .mean()
    )

    df_well[f"water_ewma_{span}"] = (
        df_well["BORE_WAT_VOL"]
        .ewm(span=span)
        .mean()
    )

In [328]:
# --------------------------------------
# 07 - expanding windows
# --------------------------------------
# baseline historico completo
# --------------------------------------

df_well["oil_expanding_mean"] = (
    df_well["BORE_OIL_VOL"]
    .expanding()
    .mean()
)

df_well["oil_expanding_std"] = (
    df_well["BORE_OIL_VOL"]
    .expanding()
    .std()
)

df_well["gas_expanding_mean"] = (
    df_well["BORE_GAS_VOL"]
    .expanding()
    .mean()
)

df_well["gas_expanding_std"] = (
    df_well["BORE_GAS_VOL"]
    .expanding()
    .std()
)

df_well["water_expanding_mean"] = (
    df_well["BORE_WAT_VOL"]
    .expanding()
    .mean()
)

df_well["water_expanding_std"] = (
    df_well["BORE_WAT_VOL"]
    .expanding()
    .std()
)

In [329]:
# --------------------------------------
# 08 - cumulative features
# --------------------------------------
# producao acumulada
# maturidade do poco
# --------------------------------------

df_well["oil_cumulative"] = (
    df_well["BORE_OIL_VOL"]
    .cumsum()
)

df_well["gas_cumulative"] = (
    df_well["BORE_GAS_VOL"]
    .cumsum()
)

df_well["water_cumulative"] = (
    df_well["BORE_WAT_VOL"]
    .cumsum()
)

In [330]:
# --------------------------------------
# 09 - velocidade e aceleracao
# --------------------------------------
# dinamica operacional
# --------------------------------------

df_well["oil_velocity"] = (
    df_well["oil_delta_1d"]
)

df_well["oil_acceleration"] = (
    df_well["oil_delta_1d"]
    .diff()
)

df_well["gas_velocity"] = (
    df_well["gas_delta_1d"]
)

df_well["gas_acceleration"] = (
    df_well["gas_delta_1d"]
    .diff()
)


In [331]:
# --------------------------------------
# 10 - trend features
# --------------------------------------
# direcao operacinal
# --------------------------------------

df_well["oil_trend_strength"] = (
    df_well["oil_roll_mean_7"] - df_well["oil_roll_mean_30"]
)

df_well["gas_trend_strength"] = (
    df_well["gas_roll_mean_7"] - df_well["gas_roll_mean_30"]
)

df_well["water_trend_strength"] = (
    df_well["water_roll_mean_7"]
    -
    df_well["water_roll_mean_30"]
)

In [332]:
# --------------------------------------
# 11 - relative trend
# --------------------------------------
# producao relativa a tendencia historica
# --------------------------------------

df_well["oil_vs_trend"] = (
    df_well["BORE_OIL_VOL"] / df_well["oil_roll_mean_30"]
)

df_well["gas_vs_trend"] = (
    df_well["BORE_GAS_VOL"] / df_well["gas_roll_mean_30"]
)

df_well["water_vs_trend"] = (
    df_well["BORE_WAT_VOL"] / df_well["water_roll_mean_30"]
)

In [333]:
# --------------------------------------
# 12 - volatility index
# --------------------------------------
# indice de volatilidade operacional
# --------------------------------------

df_well["oil_volatility_index"] = (
    df_well["oil_roll_std_14"] / df_well["oil_roll_mean_14"]
)

df_well["gas_volatility_index"] = (
    df_well["gas_roll_std_14"] / df_well["gas_roll_mean_14"]
)

In [334]:
# --------------------------------------
# 13 - momentum features
# --------------------------------------
# forca de movimento operacional
# --------------------------------------

df_well["oil_momentum_7d"] = (
    df_well["BORE_OIL_VOL"] - df_well["oil_lag_7"]
)

df_well["oil_momentum_30d"] = (
    df_well["BORE_OIL_VOL"] - df_well["oil_lag_30"]
)

In [335]:
# --------------------------------------
# 14 - rate of change (roc)
# --------------------------------------
# taxa de mudanca temporal
# --------------------------------------

df_well["oil_roc_7d"] = (
    (
        df_well["BORE_OIL_VOL"]
        -
        df_well["oil_lag_7"]
    )
    /
    df_well["oil_lag_7"]
)

df_well["oil_roc_30d"] = (
    (
        df_well["BORE_OIL_VOL"]
        -
        df_well["oil_lag_30"]
    )
    /
    df_well["oil_lag_30"]
)

In [336]:
# --------------------------------------
# 15 - z-score temporal
# --------------------------------------
# desvio relativo da tendencia
# --------------------------------------

df_well["oil_zscore_30"] = (
    (
        df_well["BORE_OIL_VOL"]
        -
        df_well["oil_roll_mean_30"]
    )
    /
    df_well["oil_roll_std_30"]
)

In [337]:
# --------------------------------------
# 16 - remover infinitos
# --------------------------------------

df_well.replace(
    [np.inf, -np.inf],
    np.nan,
    inplace=True
)

,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,AVG_DOWNHOLE_TEMPERATURE,...,oil_vs_trend,gas_vs_trend,water_vs_trend,oil_volatility_index,gas_volatility_index,oil_momentum_7d,oil_momentum_30d,oil_roc_7d,oil_roc_30d,oil_zscore_30
DATEPRD,,,,,,,,,,,,,,,,,,,,,
2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,96.87589,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2016-09-14,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,0.0,-351.68,NaN,-1.0,-0.747232
2016-09-15,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,0.0,-360.71,NaN,-1.0,-0.694389
2016-09-16,NO 15/9-F-5 AH,5769,15/9-F-5,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,NaN,...,0.0,0.0,0.0,NaN,NaN,0.0,-357.32,NaN,-1.0,-0.642862


In [338]:
df_well.reset_index(inplace=True)

In [340]:
df_well.shape

(15634, 117)

In [344]:
for col in df_well.columns:
    print(col)

DATEPRD
WELL_BORE_CODE
NPD_WELL_BORE_CODE
NPD_WELL_BORE_NAME
NPD_FIELD_CODE
NPD_FIELD_NAME
NPD_FACILITY_CODE
NPD_FACILITY_NAME
ON_STREAM_HRS
AVG_DOWNHOLE_PRESSURE
AVG_DOWNHOLE_TEMPERATURE
AVG_DP_TUBING
AVG_ANNULUS_PRESS
AVG_CHOKE_SIZE_P
AVG_CHOKE_UOM
AVG_WHP_P
AVG_WHT_P
DP_CHOKE_SIZE
BORE_OIL_VOL
BORE_GAS_VOL
BORE_WAT_VOL
BORE_WI_VOL
FLOW_KIND
WELL_TYPE
diff_dias
oil_lag_1
gas_lag_1
water_lag_1
oil_lag_3
gas_lag_3
water_lag_3
oil_lag_7
gas_lag_7
water_lag_7
oil_lag_14
gas_lag_14
water_lag_14
oil_lag_30
gas_lag_30
water_lag_30
oil_roll_mean_3
gas_roll_mean_3
water_roll_mean_3
oil_roll_mean_7
gas_roll_mean_7
water_roll_mean_7
oil_roll_mean_14
gas_roll_mean_14
water_roll_mean_14
oil_roll_mean_30
gas_roll_mean_30
water_roll_mean_30
oil_roll_std_7
gas_roll_std_7
water_roll_std_7
oil_roll_std_14
gas_roll_std_14
water_roll_std_14
oil_roll_std_30
gas_roll_std_30
water_roll_std_30
oil_delta_1d
gas_delta_1d
water_delta_1d
oil_delta_3d
gas_delta_3d
water_delta_3d
oil_delta_7d
gas_delta_7d
water_d

In [341]:
df_well.head()

,DATEPRD,WELL_BORE_CODE,NPD_WELL_BORE_CODE,NPD_WELL_BORE_NAME,NPD_FIELD_CODE,NPD_FIELD_NAME,NPD_FACILITY_CODE,NPD_FACILITY_NAME,ON_STREAM_HRS,AVG_DOWNHOLE_PRESSURE,...,oil_vs_trend,gas_vs_trend,water_vs_trend,oil_volatility_index,gas_volatility_index,oil_momentum_7d,oil_momentum_30d,oil_roc_7d,oil_roc_30d,oil_zscore_30
0,2014-04-07,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,0.00000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-04-08,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-04-09,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-04-10,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-04-11,NO 15/9-F-1 C,7405,15/9-F-1 C,3420717,VOLVE,369304,MÆRSK INSPIRER,0.0,310.37614,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [339]:
# --------------------------------------
# 19 - salvar em arquivo
# --------------------------------------
df_well.to_csv("volve_with_feature_engineering_temporal.csv", index=False)

print("Arquivo salvo: volve_with_feature_engineering_temporal.csv")

Arquivo salvo: volve_with_feature_engineering_temporal.csv


### Dicionário de Dados — Base com Feature Engineering Temporal

| Coluna | Descrição | Tipo de Dados | Unidade de Medida | Equipamento de Medição | Natureza da Variável | Local da Medição |
|---|---|---|---|---|---|---|
| `DATEPRD` | Data da produção/operação diária | datetime | data | Historian | tempo | N/A |
| `WELL_BORE_CODE` | Código interno do poço | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_WELL_BORE_CODE` | Código oficial do poço na NPD | inteiro | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_WELL_BORE_NAME` | Nome oficial do poço na NPD | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FIELD_CODE` | Código oficial do campo | inteiro | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FIELD_NAME` | Nome do campo petrolífero | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FACILITY_CODE` | Código da instalação offshore | inteiro | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `NPD_FACILITY_NAME` | Nome da instalação/FPSO/plataforma | string | N/A | Sistema de cadastro corporativo | identificação | N/A |
| `ON_STREAM_HRS` | Horas em operação no dia | float | horas | Sistema supervisório | tempo | Poço / sistema de produção |
| `AVG_DOWNHOLE_PRESSURE` | Pressão média no fundo do poço | float | bar(a) | Gauge de fundo | pressão | Fundo do poço |
| `AVG_DOWNHOLE_TEMPERATURE` | Temperatura média no fundo do poço | float | °C | Sensor downhole | temperatura | Fundo do poço |
| `AVG_DP_TUBING` | Delta de pressão médio no tubing | float | bar | Sensor de pressão do tubing | pressão | Tubing de produção |
| `AVG_ANNULUS_PRESS` | Pressão média do anular | float | bar | Sensor do anular | pressão | Espaço anular |
| `AVG_CHOKE_SIZE_P` | Abertura média do choke | float | % | Sensor do choke | abertura | Choke de superfície |
| `AVG_CHOKE_UOM` | Unidade de medida do choke | string | texto | Sensor do choke | unidade de abertura | Choke de superfície |
| `AVG_WHP_P` | Pressão média na cabeça do poço | float | bar | Sensor wellhead | pressão | Cabeça do poço |
| `AVG_WHT_P` | Temperatura média na cabeça do poço | float | °C | Sensor wellhead | temperatura | Cabeça do poço |
| `DP_CHOKE_SIZE` | Delta de pressão associado ao choke | float | bar | Sensores do choke | pressão | Linha do choke |
| `BORE_OIL_VOL` | Volume diário de óleo produzido | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `BORE_GAS_VOL` | Volume diário de gás produzido | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `BORE_WAT_VOL` | Volume diário de água produzida | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `BORE_WI_VOL` | Volume diário de água injetada | float | Sm3/d | Medidor de injeção | vazão | Linha de injeção de água |
| `FLOW_KIND` | Tipo de fluxo/operação do poço | string | N/A | Sistema supervisório | classificação operacional | N/A |
| `WELL_TYPE` | Tipo do poço | string | N/A | Sistema de engenharia de produção | classificação operacional | N/A |
| `diff_dias` | Diferença de dias entre registros consecutivos | float | dias | Historian | tempo | N/A |
| `oil_lag_1` | Volume de óleo deslocado em 1 período anterior | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_1` | Volume de gás deslocado em 1 período anterior | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_1` | Volume de água deslocado em 1 período anterior | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_3` | Volume de óleo deslocado em 3 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_3` | Volume de gás deslocado em 3 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_3` | Volume de água deslocado em 3 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_7` | Volume de óleo deslocado em 7 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_7` | Volume de gás deslocado em 7 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_7` | Volume de água deslocado em 7 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_14` | Volume de óleo deslocado em 14 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_14` | Volume de gás deslocado em 14 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_14` | Volume de água deslocado em 14 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_lag_30` | Volume de óleo deslocado em 30 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_lag_30` | Volume de gás deslocado em 30 períodos anteriores | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_lag_30` | Volume de água deslocado em 30 períodos anteriores | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_3` | Média móvel de óleo em janela de 3 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_3` | Média móvel de gás em janela de 3 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_3` | Média móvel de água em janela de 3 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_7` | Média móvel de óleo em janela de 7 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_7` | Média móvel de gás em janela de 7 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_7` | Média móvel de água em janela de 7 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_14` | Média móvel de óleo em janela de 14 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_14` | Média móvel de gás em janela de 14 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_14` | Média móvel de água em janela de 14 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_mean_30` | Média móvel de óleo em janela de 30 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_roll_mean_30` | Média móvel de gás em janela de 30 períodos | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_roll_mean_30` | Média móvel de água em janela de 30 períodos | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_roll_std_7` | Desvio padrão móvel de óleo em 7 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_roll_std_7` | Desvio padrão móvel de gás em 7 períodos | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_roll_std_7` | Desvio padrão móvel de água em 7 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_roll_std_14` | Desvio padrão móvel de óleo em 14 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_roll_std_14` | Desvio padrão móvel de gás em 14 períodos | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_roll_std_14` | Desvio padrão móvel de água em 14 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_roll_std_30` | Desvio padrão móvel de óleo em 30 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_roll_std_30` | Desvio padrão móvel de gás em 30 períodos | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_roll_std_30` | Desvio padrão móvel de água em 30 períodos | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_delta_1d` | Variação absoluta do óleo em 1 dia | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_delta_1d` | Variação absoluta do gás em 1 dia | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_delta_1d` | Variação absoluta da água em 1 dia | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_delta_3d` | Variação absoluta do óleo em 3 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_delta_3d` | Variação absoluta do gás em 3 dias | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_delta_3d` | Variação absoluta da água em 3 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_delta_7d` | Variação absoluta do óleo em 7 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_delta_7d` | Variação absoluta do gás em 7 dias | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_delta_7d` | Variação absoluta da água em 7 dias | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_pct_change_1d` | Variação percentual do óleo em 1 dia | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `gas_pct_change_1d` | Variação percentual do gás em 1 dia | float | % | Medidor de gás | variação percentual | Linha de gás / separador |
| `water_pct_change_1d` | Variação percentual da água em 1 dia | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_pct_change_7d` | Variação percentual do óleo em 7 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `gas_pct_change_7d` | Variação percentual do gás em 7 dias | float | % | Medidor de gás | variação percentual | Linha de gás / separador |
| `water_pct_change_7d` | Variação percentual da água em 7 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_pct_change_14d` | Variação percentual do óleo em 14 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `gas_pct_change_14d` | Variação percentual do gás em 14 dias | float | % | Medidor de gás | variação percentual | Linha de gás / separador |
| `water_pct_change_14d` | Variação percentual da água em 14 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_ewma_3` | Média móvel exponencial do óleo com janela 3 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_3` | Média móvel exponencial do gás com janela 3 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_3` | Média móvel exponencial da água com janela 3 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_ewma_7` | Média móvel exponencial do óleo com janela 7 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_7` | Média móvel exponencial do gás com janela 7 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_7` | Média móvel exponencial da água com janela 7 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_ewma_14` | Média móvel exponencial do óleo com janela 14 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_14` | Média móvel exponencial do gás com janela 14 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_14` | Média móvel exponencial da água com janela 14 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_ewma_30` | Média móvel exponencial do óleo com janela 30 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `gas_ewma_30` | Média móvel exponencial do gás com janela 30 | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `water_ewma_30` | Média móvel exponencial da água com janela 30 | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_expanding_mean` | Média acumulada expandida do óleo | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `oil_expanding_std` | Desvio padrão acumulado expandido do óleo | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `gas_expanding_mean` | Média acumulada expandida do gás | float | Sm3/d | Medidor de gás | vazão | Linha de gás / separador |
| `gas_expanding_std` | Desvio padrão acumulado expandido do gás | float | Sm3/d | Medidor de gás | dispersão | Linha de gás / separador |
| `water_expanding_mean` | Média acumulada expandida da água | float | Sm3/d | Medidor multifásico | vazão | Linha de produção / separador |
| `water_expanding_std` | Desvio padrão acumulado expandido da água | float | Sm3/d | Medidor multifásico | dispersão | Linha de produção / separador |
| `oil_cumulative` | Volume acumulado de óleo ao longo do tempo | float | Sm3 | Medidor multifásico | volume | Linha de produção / separador |
| `gas_cumulative` | Volume acumulado de gás ao longo do tempo | float | Sm3 | Medidor de gás | volume | Linha de gás / separador |
| `water_cumulative` | Volume acumulado de água ao longo do tempo | float | Sm3 | Medidor multifásico | volume | Linha de produção / separador |
| `oil_velocity` | Velocidade de variação do óleo entre períodos | float | Sm3/dia | Medidor multifásico | velocidade | Linha de produção / separador |
| `oil_acceleration` | Aceleração da variação do óleo entre períodos | float | Sm3/dia² | Medidor multifásico | aceleração | Linha de produção / separador |
| `gas_velocity` | Velocidade de variação do gás entre períodos | float | Sm3/dia | Medidor de gás | velocidade | Linha de gás / separador |
| `gas_acceleration` | Aceleração da variação do gás entre períodos | float | Sm3/dia² | Medidor de gás | aceleração | Linha de gás / separador |
| `oil_trend_strength` | Intensidade da tendência do óleo frente à média móvel | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `gas_trend_strength` | Intensidade da tendência do gás frente à média móvel | float | adimensional | Medidor de gás | tendência | Linha de gás / separador |
| `water_trend_strength` | Intensidade da tendência da água frente à média móvel | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `oil_vs_trend` | Razão entre óleo observado e tendência estimada | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `gas_vs_trend` | Razão entre gás observado e tendência estimada | float | adimensional | Medidor de gás | tendência | Linha de gás / separador |
| `water_vs_trend` | Razão entre água observada e tendência estimada | float | adimensional | Medidor multifásico | tendência | Linha de produção / separador |
| `oil_volatility_index` | Índice relativo de volatilidade do óleo | float | adimensional | Medidor multifásico | volatilidade | Linha de produção / separador |
| `gas_volatility_index` | Índice relativo de volatilidade do gás | float | adimensional | Medidor de gás | volatilidade | Linha de gás / separador |
| `oil_momentum_7d` | Momentum do óleo em 7 dias | float | Sm3/d | Medidor multifásico | momentum | Linha de produção / separador |
| `oil_momentum_30d` | Momentum do óleo em 30 dias | float | Sm3/d | Medidor multifásico | momentum | Linha de produção / separador |
| `oil_roc_7d` | Taxa de variação do óleo em 7 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_roc_30d` | Taxa de variação do óleo em 30 dias | float | % | Medidor multifásico | variação percentual | Linha de produção / separador |
| `oil_zscore_30` | Z-score do óleo em janela de 30 períodos | float | adimensional | Medidor multifásico | desvio padronizado | Linha de produção / separador |
